<a href="https://colab.research.google.com/github/SuCh50-Coder/Susnata/blob/main/StackerPy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# StackerPy v1.0
## Pranjal Sengupta

<a target="_blank" href="https://colab.research.google.com/github/pranjalsg/astro/blob/main/StackerPy.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This is a simple version of an Astronomical Image Stacking Software build entirely using Python. Use any camera to record the video of the object through the telescope (either by using a mobile adapter and imaging through the lens or making the video by a dedicated astronomy camera).

This notebook is written primarily for planetary imaging (including the Moon, the Sun etc.) as many popular advanced stacking software exists for Deep Sky Astrophotography. However, a deep sky stacking mode is also included in this (<span style="background-color:pink; color:black">untested</span> - if anyone is willing to contribute and test, feel free to do so and let me know your feedback at pranjalsengupta5@gmail.com .

<span style="background-color:red; color:white">WARNING:</span> <span style="background-color:yellow; color:red">Never point your telescope or camera towards the Sun without a proper Solar filter.</span>


### Stacking Basics

Stacking involves the following general steps:

- __Extracting and Reading Frames__: The video is separated into the individual frames and read into memory.
- __Quality Analysis__: The frames are ranked based on their quality and only the top few frames get selected. Images with sharp edges signifying better quality get selected for stacking.
- __Alignment__: The images are aligned based on extended objects (planetary) or star patterns (deep-sky) to prepare them for stacking.
- __Stacking__: The aligned images are stacked through the pixelwise calculation of median or mean.

Post-processing may be done to enhance the stacked image (eg. using Wavelet)

### Using this notebook

Remove the # in the second line of the next cell and run the cell if you are running this software for the first time or in Google Colab.(Shift + Enter). Otherwise just remove the asterisk from the second line and click on `Run All` in the menu bar if you are using Google Colab (`Run` -> `Run All Cells` on Jupyter Notebook) if you just want to use the software and then go the the end of this file to find the Actual Interface!

In [3]:
# Remove the # in the next line and run this cell if you are running this software for the first time or in Google Colab (Shift + Enter).
!pip install opencv-python numpy matplotlib scipy astroalign ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 69.8 MB/s eta 0:00:00


Run this cell below (Shift + Enter) to import the required libraries.

In [4]:
# importing libraries

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
import astroalign as aa
import scipy.ndimage as ndimage
from tqdm.notebook import tqdm
import concurrent.futures
import ipywidgets as widgets
from IPython.display import display, clear_output

## Functions and Jargons

Feel free to change anything here, just do Ctrl/CMD + Z if you mess up! Otheriwse, no changes are required here, just run the following cell (Shift + Enter).

In [5]:
def extract_and_score_video(video_path, max_frames=500):
    cap = cv2.VideoCapture(video_path)
    frames = []
    scores = []

    with tqdm(total=max_frames, desc="Reading Video") as pbar:
        count = 0
        while True:
            ret, frame = cap.read()
            if not ret or count >= max_frames:
                break

            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()

            frames.append(frame)
            scores.append(sharpness)
            count += 1
            pbar.update(1)

    cap.release()
    sorted_indices = np.argsort(scores)[::-1]
    return [frames[i] for i in sorted_indices]

def _score_single_image(file_path):
    """Helper function for parallel execution"""
    frame = cv2.imread(file_path)
    if frame is None:
        return None, -1
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()
    return frame, sharpness

def load_and_score_directory(dir_path, max_frames=500):
    valid_exts = ['*.png', '*.jpg', '*.jpeg', '*.tif', '*.tiff']
    image_files = []
    for ext in valid_exts:
        image_files.extend(glob.glob(os.path.join(dir_path, ext)))

    total_files = min(len(image_files), max_frames)
    frames = []
    scores = []

    # Process images in parallel using ThreadPoolExecutor
    with concurrent.futures.ThreadPoolExecutor() as executor:
        results = list(tqdm(executor.map(_score_single_image, image_files[:total_files]),
                            total=total_files, desc="Parallel Reading Directory"))

    # Unpack the results
    for frame, score in results:
        if frame is not None:
            frames.append(frame)
            scores.append(score)

    sorted_indices = np.argsort(scores)[::-1]
    print(f"Loaded and scored {len(frames)} images from directory.")
    return [frames[i] for i in sorted_indices]

def align_frames(frames, top_percentage=20, target_type='Planetary'):
    num_keep = max(1, int(len(frames) * (top_percentage / 100)))
    best_frames = frames[:num_keep]

    reference_frame = best_frames[0]
    ref_gray = cv2.cvtColor(reference_frame, cv2.COLOR_BGR2GRAY)

    print(f"Parallel aligning the top {num_keep} frames using {target_type} mode...")

    # Helper function executed by the worker threads
    def align_single(img):
        if target_type == 'Planetary':
            img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            warp_matrix = np.eye(2, 3, dtype=np.float32)
            criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 50, 1e-3)

            try:
                _, warp_matrix = cv2.findTransformECC(ref_gray, img_gray, warp_matrix, cv2.MOTION_EUCLIDEAN, criteria)
                return cv2.warpAffine(img, warp_matrix, (img.shape[1], img.shape[0]), flags=cv2.INTER_LINEAR + cv2.WARP_INVERSE_MAP)
            except cv2.error:
                return None

        elif target_type == 'Deep Sky':
            try:
                aligned_img, _ = aa.register(img, reference_frame)
                return aligned_img
            except Exception:
                return None

    aligned_frames = [reference_frame]

    # Use ThreadPoolExecutor to run alignments in parallel
    with concurrent.futures.ThreadPoolExecutor() as executor:
        # We slice [1:] because the first frame is our reference frame
        futures = executor.map(align_single, best_frames[1:])

        for result in tqdm(futures, total=num_keep-1, desc="Aligning Frames"):
            if result is not None:
                aligned_frames.append(result)

    return aligned_frames

def stack_frames(aligned_frames, method='median'):
    stack = np.stack(aligned_frames, axis=0).astype(np.float32)
    integrated = np.zeros_like(aligned_frames[0], dtype=np.float32)
    height = stack.shape[1]
    chunk_size = max(1, height // 100)

    for y in tqdm(range(0, height, chunk_size), desc=f"Stacking ({method})"):
        y_end = min(y + chunk_size, height)
        if method == 'median':
            integrated[y:y_end] = np.median(stack[:, y:y_end], axis=0)
        elif method == 'mean':
            integrated[y:y_end] = np.mean(stack[:, y:y_end], axis=0)

    return np.clip(integrated, 0, 255).astype(np.uint8)

def multi_scale_wavelet_sharpen(img, s1=0.0, s2=0.0, s3=0.0):
    if img is None:
        return None

    img_float = img.astype(np.float32)

    # Layer 1: Fine details (e.g., sharp planetary rims, small craters, fine festoons)
    g1 = ndimage.gaussian_filter(img_float, sigma=1.0)
    layer1_detail = img_float - g1

    # Layer 2: Medium details (e.g., cloud bands, mid-sized craters)
    g2 = ndimage.gaussian_filter(img_float, sigma=2.5)
    layer2_detail = g1 - g2

    # Layer 3: Coarse details (e.g., global contrast, large cloud regions)
    g3 = ndimage.gaussian_filter(img_float, sigma=5.0)
    layer3_detail = g2 - g3

    # Recombine by applying individual layer strengths
    sharpened = img_float + (layer1_detail * s1) + (layer2_detail * s2) + (layer3_detail * s3)

    return np.clip(sharpened, 0, 255).astype(np.uint8)

def launch_astro_dashboard():
    # --- 1. Local State Management ---
    app_state = {
        'stacked_image': None,
        'final_image': None,
        'aligned_count': 0
    }

    # --- 2. Create UI Elements ---
    ui_title = widgets.HTML(value="<h2>Astro-Stacker & Processor</h2>")

    input_type = widgets.RadioButtons(options=['Video File', 'Directory of Images'], description='Input Type:', value='Video File')
    file_input = widgets.Text(description="Path:", value="video.avi")
    target_type = widgets.RadioButtons(options=['Planetary', 'Deep Sky'], description='Target:', value='Planetary')
    stack_method = widgets.Dropdown(options=['median', 'mean'], value='median', description='Method:')
    quality_slider = widgets.IntSlider(min=1, max=100, step=1, value=20, description='Keep Top %:')

    stack_button = widgets.Button(description="Align & Stack Frames", button_style='primary', layout=widgets.Layout(width='95%'))

    wavelet_s1 = widgets.FloatSlider(min=0.0, max=5.0, step=0.1, value=0.0, description='Layer 1 (Fine):', disabled=True, style={'description_width': 'initial'})
    wavelet_s2 = widgets.FloatSlider(min=0.0, max=5.0, step=0.1, value=0.0, description='Layer 2 (Med):', disabled=True, style={'description_width': 'initial'})
    wavelet_s3 = widgets.FloatSlider(min=0.0, max=5.0, step=0.1, value=0.0, description='Layer 3 (Gross):', disabled=True, style={'description_width': 'initial'})

    save_input = widgets.Text(description="Output Name:", value="astro_output.png")
    save_button = widgets.Button(description="Save Image (PNG)", button_style='warning', disabled=True, icon='save')

    output_stacking = widgets.Output()
    output_plot = widgets.Output()

    # --- 3. Callbacks and Logic ---
    def render_live_view(*args):
        if app_state['stacked_image'] is None:
            return

        with output_plot:
            clear_output(wait=True)

            final_image = multi_scale_wavelet_sharpen(
                app_state['stacked_image'],
                s1=wavelet_s1.value,
                s2=wavelet_s2.value,
                s3=wavelet_s3.value
            )

            app_state['final_image'] = final_image
            save_button.disabled = False

            fig, axes = plt.subplots(1, 2, figsize=(16, 8))

            axes[0].imshow(cv2.cvtColor(app_state['stacked_image'], cv2.COLOR_BGR2RGB))
            axes[0].set_title(f"Raw Stack ({app_state['aligned_count']} frames)")
            axes[0].axis('off')

            axes[1].imshow(cv2.cvtColor(final_image, cv2.COLOR_BGR2RGB))
            axes[1].set_title(f"Wavelet Live Adjust (L1: {wavelet_s1.value} | L2: {wavelet_s2.value} | L3: {wavelet_s3.value})")
            axes[1].axis('off')

            plt.tight_layout()
            plt.show()

    def toggle_observers(observe=True):
        for slider in [wavelet_s1, wavelet_s2, wavelet_s3]:
            if observe:
                slider.observe(render_live_view, names='value')
            else:
                try:
                    slider.unobserve(render_live_view, names='value')
                except ValueError:
                    pass

    def on_stack_clicked(b):
        toggle_observers(observe=False)

        wavelet_s1.value = 0.0
        wavelet_s2.value = 0.0
        wavelet_s3.value = 0.0
        save_button.disabled = True

        with output_stacking:
            clear_output()

            if input_type.value == 'Video File':
                print("Step 1: Analyzing video file properties...")
                cap = cv2.VideoCapture(file_input.value)
                max_video_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                cap.release()

                if max_video_frames <= 0:
                    print("Warning: Could not read video frame metadata. Defaulting to 500 frames limit.")
                    max_video_frames = 500
                else:
                    print(f"Detected a total of {max_video_frames} frames in video.")

                print("Reading and Scoring frames...")
                frames = extract_and_score_video(file_input.value, max_frames=max_video_frames)
            else:
                print("Step 1: Reading and Scoring frames from Directory...")
                frames = load_and_score_directory(file_input.value, max_frames=2000)

            if not frames:
                print("Error: No frames loaded. Check your path.")
                return

            print("\nStep 2: Aligning frames...")
            aligned = align_frames(frames, top_percentage=quality_slider.value, target_type=target_type.value)

            print("\nStep 3: Stacking...")
            app_state['stacked_image'] = stack_frames(aligned, method=stack_method.value)
            app_state['aligned_count'] = len(aligned)

            wavelet_s1.disabled = False
            wavelet_s2.disabled = False
            wavelet_s3.disabled = False
            print("\nStacking complete! Adjust the sliders below to sharpen live.")

        toggle_observers(observe=True)
        render_live_view()

    def on_save_clicked(b):
        if app_state['final_image'] is None:
            return

        filename = save_input.value.strip()
        if not filename.lower().endswith('.png'):
            filename += '.png'

        success = cv2.imwrite(filename, app_state['final_image'])

        with output_stacking:
            if success:
                print(f"Successfully saved processed image to: {os.path.abspath(filename)}")
            else:
                print(f"Error: Could not save image to {filename}. Check directory permissions.")

    # --- 4. Wire up the click callbacks ---
    stack_button.on_click(on_stack_clicked)
    save_button.on_click(on_save_clicked)

    # --- 5. UI Layout Construction & Display ---
    left_controls = widgets.VBox([input_type, file_input, quality_slider])
    right_controls = widgets.VBox([target_type, stack_method, widgets.HTML("<hr><b>Wavelet Layers</b>"), wavelet_s1, wavelet_s2, wavelet_s3])
    save_box = widgets.HBox([save_input, save_button], layout=widgets.Layout(margin='15px 0px 0px 0px'))

    dashboard_layout = widgets.VBox([
        ui_title,
        widgets.HBox([left_controls, right_controls]),
        stack_button,
        save_box,
        output_stacking,
        output_plot
    ])

    display(dashboard_layout)

## Graphical Interface for Ease of Use

First run the following cell to load the Interface.

Then specify how you want to input the frames/video. If you are inputting a video, make sure to add the extension. Otherwise if your frames are preloaded, put them into a folder and specify the folder name (`Directory of Images`). For ease of use, make sure the video/directory is in the same folder as this notebook (or accessible by Colab).

Use the `Target` toggle appropriately and the `Keep Top %` which selects the fraction of the best images being considered for stacking. (Default = 20%)

The Median method is generally considered superior to Mean as it rejects objects present in only 1 or 2 frames (non-persistent)

Once these options are selected, click on `Align & Stack Frames`

Once done and the image is visible, play around with the `Wavelet Layers`and finally click on `Save Image (PNG)` with a desired output filename and you're done!

In [6]:
launch_astro_dashboard()

In [8]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful